# 03 · Mensajería avanzada: ACK con timeout, historial, RTT

**Objetivo**: añadir tres capacidades sobre la mensajería básica del notebook 02.

Fuentes: [src/main.ts](../../src/main.ts), [src/history.ts](../../src/history.ts), [src/protocol.ts](../../src/protocol.ts) (PING/PONG).

## 1. Acuse de recibo (CHAT_ACK) con timeout

Mandar un CHAT por TCP **no garantiza** que el peer lo haya leído. TCP garantiza entrega al kernel del receptor, pero:

- el proceso del receptor puede haber muerto antes de leerlo
- el receptor podría descartarlo por error
- el socket podría estar cerrándose

Para tener confirmación **a nivel de aplicación**, añadimos un mensaje `CHAT_ACK { messageId }` que el receptor envía tras procesar el CHAT.

### Carrera entre ACK y timeout

El emisor:

1. Genera `messageId` único, manda CHAT.
2. Programa `setTimeout(3000)` que dirá ✗ si dispara.
3. Guarda en `pendingAcks[messageId] = {timer, peerId}`.
4. Cuando llega CHAT_ACK → cancela el timer y pinta ✓.

Diagrama:

```
tiempo →

A: send CHAT id=abc          
   timer(3s) armed                                  
                                              B: render CHAT
                                                 send CHAT_ACK id=abc
A: clearTimeout → ✓                  

─────────────────────────────  o si B no responde:

A: send CHAT id=xyz
   timer(3s) armed
                                              (B silencioso)
A: timer fires → ✗ no entregado
```

In [ ]:
# Simulación de la carrera ACK vs timeout
import asyncio
import random

ACK_TIMEOUT = 3.0  # s

async def send_with_ack(message_id, network_delay, receiver_alive):
    pending = asyncio.get_event_loop().create_future()
    async def fake_remote():
        await asyncio.sleep(network_delay)
        if receiver_alive and not pending.done():
            pending.set_result('ack')
    asyncio.create_task(fake_remote())
    try:
        await asyncio.wait_for(pending, timeout=ACK_TIMEOUT)
        print(f'  {message_id}: ✓ entregado en {network_delay:.1f}s')
    except asyncio.TimeoutError:
        print(f'  {message_id}: ✗ no entregado (timeout)')

async def main():
    print('Caso 1: red rápida, receptor vivo')
    await send_with_ack('m1', 0.1, True)
    print('\nCaso 2: red lenta pero dentro del timeout')
    await send_with_ack('m2', 2.5, True)
    print('\nCaso 3: receptor muerto')
    await send_with_ack('m3', 1.0, False)
    print('\nCaso 4: red más lenta que el timeout')
    await send_with_ack('m4', 5.0, True)

await main()

### Variante con librería: `websockets` (PING/PONG nativo + RTT real)

```
pip install websockets
```

El protocolo WebSocket define **frames de control PING/PONG** (opcodes `0x9`/`0xA`) que el stack maneja automáticamente. `ws.ping()` devuelve un `Future` que se resuelve en cuanto llega el `PONG` correspondiente — exactamente la carrera ACK/timeout que simulamos arriba, pero gestionada por la librería. No hay que programar `pendingAcks` ni `clearTimeout`.

```python
import asyncio
import websockets
import time

# ─── Servidor (peer B) ───────────────────────────────────────────────────────
async def handler(ws):
    # websockets responde a PING automáticamente con PONG.
    # Solo hay que mantener la conexión viva.
    async for _ in ws:
        pass

async def server():
    async with websockets.serve(handler, "0.0.0.0", 8765):
        await asyncio.Future()

# ─── Cliente (peer A) — medición de RTT con ACK built-in ────────────────────
async def measure_rtt(uri, n=5):
    async with websockets.connect(uri) as ws:
        rtts = []
        for i in range(n):
            t0 = time.perf_counter()
            pong_waiter = await ws.ping()    # envía PING frame
            await pong_waiter                # espera PONG (timeout configurable)
            rtt_ms = (time.perf_counter() - t0) * 1000
            rtts.append(rtt_ms)
            print(f"PING {i+1}: RTT = {rtt_ms:.2f} ms")
            await asyncio.sleep(1)
        avg = sum(rtts) / len(rtts)
        print(f"RTT promedio: {avg:.2f} ms")

# asyncio.run(server())               # peer B
# asyncio.run(measure_rtt("ws://IP_PEER_B:8765"))  # peer A
```

**Timeout de ACK** — `ws.ping()` acepta un timeout:

```python
pong_waiter = await ws.ping()
try:
    await asyncio.wait_for(pong_waiter, timeout=3.0)
    print("✓ entregado")
except asyncio.TimeoutError:
    print("✗ no entregado")
    await ws.close()
```

**CHAT ACK a nivel de aplicación** — PING/PONG mide liveness de la conexión. Para ACK de mensaje de chat específico (con `messageId`), el patrón es el mismo pero sobre el canal de datos:

```python
# Emisor
pending: dict[str, asyncio.Future] = {}

async def send_chat(ws, msg_id, text):
    future = asyncio.get_event_loop().create_future()
    pending[msg_id] = future
    await ws.send(json.dumps({"type": "CHAT", "id": msg_id, "text": text}))
    try:
        await asyncio.wait_for(future, timeout=3.0)
        print(f"{msg_id}: ✓")
    except asyncio.TimeoutError:
        print(f"{msg_id}: ✗")

# Receptor
async def on_message(ws, raw):
    msg = json.loads(raw)
    if msg["type"] == "CHAT":
        await ws.send(json.dumps({"type": "CHAT_ACK", "id": msg["id"]}))
    elif msg["type"] == "CHAT_ACK":
        if msg["id"] in pending:
            pending.pop(msg["id"]).set_result(None)
```

Ventaja sobre la implementación propia: WebSocket garantiza el framing y la entrega ordenada; solo hay que gestionar la lógica de ACK a nivel de aplicación.

### Decisión de diseño: ACK inmediato

En [src/main.ts](../../src/main.ts), el receptor manda el ACK **antes** de persistir en historial:

```ts
case MSG.CHAT: {
  cliSinks.chat?.(from, msg.text);                           // 1. render
  transport.send(from, { type: MSG.CHAT_ACK, messageId });  // 2. ACK ya
  void histAppend({...});                                    // 3. persistir
}
```

**¿Por qué ACK antes de persistir?** Porque el render ya implica "el humano vio el mensaje". Si el `histAppend` falla por disco lleno, no es razón para que el emisor vea ✗.

El trade-off opuesto sería "at-least-write-then-ack" (como Kafka): garantiza durabilidad pero suma latencia. En p2p-chat queremos snappy.

## 2. Historial: JSON Lines

Persistencia mínima viable: cada mensaje (in/out) se appendea como UNA línea JSON en `./history.jsonl`.

```json
{"ts":1730000000000,"dir":"out","peerId":"abc123...","messageId":"d4...","text":"hola"}
{"ts":1730000000050,"dir":"in", "peerId":"abc123...","messageId":"e5...","text":"qué tal"}
```

### Por qué JSONL

| Aspecto | JSONL | SQLite | Fichero con índice |
|---------|-------|--------|---------------------|
| Append O(1) | ✅ | ✅ (con WAL) | depende |
| Inspección con `cat`/`grep` | ✅ | ❌ (binario) | ✅ |
| Consulta por id | ❌ (scan) | ✅ (índice) | ✅ |
| Recuperación tras crash | OK (líneas truncadas se ignoran) | OK | frágil |
| Dependencias | Ninguna | sqlite | Ninguna |

Para un chat pequeño: JSONL gana en pedagogía y portabilidad. Para 10M de mensajes: SQLite o un kv store.

### `tail(n)` ingenuo

Nuestra implementación lee el fichero entero, splitea por líneas y se queda con las últimas N. Funciona para historiales pequeños.

**Reto avanzado**: para historiales grandes, lee desde el final retrocediendo en bloques hasta acumular N saltos de línea. Es lo que hace `tail -n` en Unix.

## 3. PING/PONG y medición de RTT

Cada 5s, por cada peer conectado, mandamos `PING { nonce }`. El receptor responde `PONG { nonce }`. El emisor mide:

$$RTT = t_{\text{ahora}} - t_{\text{envío del nonce}}$$

### Por qué nonce y no timestamp

Podríamos meter el `ts` en el PING y que el PONG lo devuelva, pero entonces el emisor confía en datos que viajaron y volvieron. **Nonce** es un id opaco; el emisor guarda `nonces[nonce] = ts_envio` localmente y el receptor solo necesita devolver el mismo nonce. No hace falta sincronizar relojes ni confiar en el PONG.

## 4. Media móvil exponencial (EWMA)

Una sola medida de RTT es ruidosa. Lo que queremos es un valor **suavizado** que represente el RTT *típico*. La fórmula:

$$R_t = \alpha \cdot x_t + (1-\alpha) \cdot R_{t-1}$$

donde $x_t$ es la medida cruda y $R_t$ el RTT suavizado.

### ¿Por qué "exponencial"?

Si desarrollas hacia atrás:

$$R_t = \alpha x_t + \alpha(1-\alpha)x_{t-1} + \alpha(1-\alpha)^2 x_{t-2} + ...$$

Cada medida pasada contribuye con un factor que **decae geométricamente**. Es como una media móvil de ventana infinita pero con memoria que se desvanece — sin guardar un buffer.

### Trade-off de α

| α | Comportamiento |
|---|---------------|
| 1.0 | $R = x$, sin suavizado |
| 0.5 | Reactivo, ruidoso |
| 0.2 | Punto medio — usado en p2p-chat |
| 0.125 | Conservador — el que usa TCP (SRTT) |
| 0.0 | $R = R$, no actualiza nunca |

In [ ]:
import matplotlib.pyplot as plt
import random

random.seed(42)

# Simulamos RTT con un mean shift en t=50 (la red empeora) + jitter
true_rtt = [40 if t < 50 else 120 for t in range(100)]
measured  = [r + random.gauss(0, 15) for r in true_rtt]

def ewma(xs, alpha):
    out, r = [], None
    for x in xs:
        r = x if r is None else alpha*x + (1-alpha)*r
        out.append(r)
    return out

plt.figure(figsize=(10, 5))
plt.plot(measured, '.', alpha=0.3, label='RTT medido (crudo)')
plt.plot(true_rtt, 'k--', alpha=0.5, label='RTT real')
for a in [0.05, 0.2, 0.5]:
    plt.plot(ewma(measured, a), label=f'EWMA α={a}')
plt.xlabel('PING #'); plt.ylabel('RTT (ms)'); plt.legend(); plt.grid(alpha=0.3)
plt.title('EWMA: trade-off reactividad vs estabilidad')
plt.show()

**Observa**:

- α=0.05 es muy estable pero tarda mucho en alcanzar el nuevo nivel tras el cambio en t=50.
- α=0.5 reacciona rápido pero copia gran parte del ruido.
- α=0.2 es el equilibrio razonable.

### Decisión: ¿quién marca al peer como muerto?

En p2p-chat el PING/PONG **NO** marca peers como muertos. Si el PONG no llega, el RTT no se actualiza pero el peer sigue "vivo". La fuente de verdad para liveness sigue siendo el descubrimiento UDP (timeout 10s).

Razón: separar **conectividad** (descubrimiento) de **calidad de la conexión** (RTT). Un peer puede estar saturado y tardar 5s en responder un PING pero seguir vivo.

## 5. Siguiente paso

Mensajería rica: ✓ entregado, historial, RTT por peer. Todo TCP.

Pero TCP **no sirve** para audio en tiempo real. Una retransmisión = parón audible. Necesitamos otro stack: WebRTC sobre UDP. Eso en el notebook 04.